<a href="https://colab.research.google.com/github/erickaylas-zen/FGD_2026/blob/main/LAB_D10_FGD_EAYLAS_2026_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATORIO DIRIGIDO N.° 10
## Migración de datos y ETL en Python: Caso Farmacia MediSur

**Curso:** Fundamentos de Gestión de Datos
**Semana:** 10 — Migración de Datos y ETL
**Tema:** Pipeline ETL con Python, Pandas y SQLite sobre la base oficial del proyecto
**Docente:** Pilar Rocío Sayán Mejía
**Duración estimada:** 1 hora y 40 minutos

---

> **Importante — esta base te acompaña todo el proyecto (semanas 10 a 16).**
> Aquí **no se inventan datos**. Trabajarás con la **base oficial de tu caso**, alojada en GitHub. Es la misma base que documentarás (S11), analizarás (S12), medirás (S13) y gobernarás (S14–15) hasta el proyecto final PMD2. Lo que limpies hoy es el punto de partida de todo lo que sigue.

## Caso introductorio

Eres analista de datos en **Farmacia MediSur**, una cadena de farmacias con varias sucursales. Cada sucursal registró sus ventas y clientes de manera distinta y el sistema antiguo acumuló errores. Tu tarea de esta semana **no** es construir todo el proyecto final, sino realizar una **primera migración controlada** de la base oficial:

1. **Descargar** la base oficial del caso desde el repositorio (GitHub).
2. **Extraer** los datos desde SQLite.
3. **Diagnosticar** problemas de calidad e integridad.
4. **Transformar** y limpiar los datos con funciones de Pandas.
5. **Cargar** los datos limpios en una nueva base SQLite.
6. **Unificar** una segunda fuente que llega con columnas distintas.
7. **Automatizar** la carga con una función reutilizable (stored procedure simulado).
8. **Validar** la migración completa con un checklist.

Este laboratorio trabaja solamente los temas de la **Semana 10: migración de datos y ETL**.

## Actividad 1: conceptos previos

Completa con tus propias palabras:

| Concepto | Respuesta |
|---|---|
| ¿Qué significa ETL? | Es la sigla de Extract, Transform, Load: el proceso de extraer datos de una fuente origen, transformarlos (limpiarlos, estandarizarlos, corregirlos) y cargarlos en una base o sistema destino. |
| ¿Qué ocurre en la fase Extract? | Se conecta a la fuente de datos (en este caso la base SQLite legacy) y se extraen las tablas necesarias, en nuestro caso `clientes` y `operaciones`, sin modificar aún nada del origen. |
| ¿Qué ocurre en la fase Transform? | Se diagnostican y corrigen los problemas de calidad e integridad detectados: duplicados, nulos, formatos inconsistentes, montos negativos y registros huérfanos, usando funciones de pandas. |
| ¿Qué ocurre en la fase Load? | Los datos ya limpios y transformados se cargan en una base de datos nueva (`farmacia_migrada.db`), dejando la base original intacta como respaldo. |
| ¿Por qué se valida una migración? | Porque un pipeline que corre sin errores no garantiza que los datos migrados sean correctos; hay que comparar conteos, nulos, duplicados y montos entre origen y destino para confirmar que ningún dato se perdió o corrompió.

## Actividad 2: desarrollo práctico en Colab

### Paso 1: importar librerías
Cargamos las librerías del sílabo: `sqlite3` para las bases, `pandas` para transformar y `numpy` para apoyo numérico.

In [1]:
# Paso 1: importar librerias
import sqlite3
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
np.random.seed(10)

try:
    display
except NameError:
    display = print

print("Entorno listo para el Laboratorio D10 - ETL con Farmacia MediSur")

Entorno listo para el Laboratorio D10 - ETL con Farmacia MediSur


### Paso 2: descargar la base OFICIAL del caso desde GitHub
La base **ya existe**: es la base oficial de tu caso publicada en el repositorio. La descargamos una sola vez.

> Cambia `caso` por la carpeta de **tu** caso asignado. El nombre debe coincidir **exactamente** con la carpeta del repositorio.

In [2]:
# Paso 2: descargar la base oficial del caso (una sola vez)
caso = "03_farmacia_medisur"   # <-- cambia por la carpeta de TU caso asignado
DB_ORIGEN = "farmacia_legacy.db"
DB_DESTINO = "farmacia_migrada.db"
url = f"https://raw.githubusercontent.com/Rociosayan/-PMD2_FGD_Bases_Oficiales_FDG/main/casos/{caso}/{caso}.db"

if not os.path.exists(DB_ORIGEN):
    import requests
    r = requests.get(url)
    if r.status_code == 200 and r.content[:16] == b"SQLite format 3\x00":
        with open(DB_ORIGEN, "wb") as f:
            f.write(r.content)
        print("Base oficial descargada desde GitHub:", len(r.content), "bytes")
    else:
        from google.colab import files
        print("No se pudo descargar. Sube manualmente el .db de tu caso:")
        subida = files.upload()
        with open(DB_ORIGEN, "wb") as f:
            f.write(list(subida.values())[0])
else:
    print("La base ya estaba disponible en el entorno:", DB_ORIGEN)

if os.path.exists(DB_DESTINO):
    os.remove(DB_DESTINO)
print("Base origen :", DB_ORIGEN)
print("Base destino:", DB_DESTINO)

Base oficial descargada desde GitHub: 344064 bytes
Base origen : farmacia_legacy.db
Base destino: farmacia_migrada.db


### Paso 3: EXTRACT — explorar la base y extraer la tabla de clientes
La base oficial es **relacional**: tiene varias tablas conectadas (sedes, empleados, clientes, productos/servicios, operaciones, detalle, pagos, incidencias). Primero vemos qué hay y cuántos registros trae cada tabla; luego extraemos la tabla `clientes`, que será el centro de nuestra limpieza.

In [3]:
# Paso 3: EXTRACT - explorar tablas y extraer clientes
conn_origen = sqlite3.connect(DB_ORIGEN)

tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn_origen)
print("Tablas en la base oficial:")
for t in tablas["name"]:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn_origen)["n"][0]
    print(f"   - {t:22s} {n:5d} registros")

df_clientes = pd.read_sql_query("SELECT * FROM clientes", conn_origen)
print("\nRegistros extraidos de clientes:", len(df_clientes))
print("Columnas:", list(df_clientes.columns))
display(df_clientes.head())

Tablas en la base oficial:
   - clientes                 208 registros
   - detalle_operacion       2572 registros
   - empleados                 40 registros
   - incidencias              155 registros
   - operaciones             1010 registros
   - pagos                   1000 registros
   - productos_servicios       30 registros
   - sedes                      6 registros

Registros extraidos de clientes: 208
Columnas: ['id_cliente', 'codigo_cliente', 'tipo_documento', 'num_documento', 'nombres', 'apellidos', 'correo', 'telefono', 'distrito', 'segmento', 'fecha_registro', 'condicion_cronica']


,id_cliente,codigo_cliente,tipo_documento,num_documento,nombres,apellidos,correo,telefono,distrito,segmento,fecha_registro,condicion_cronica
0,1,C0001,RUC,79941052,Iván,Aguirre Bautista,iván1@correo.com,923380330,Rímac,Cliente frecuente,2026-01-24,Hipertensión
1,2,C0002,DNI,377593,Melissa,Torres Lévano,melissa2@correo.com,966971049,Villa El Salvador,Paciente crónico,2026-05-18,Asma
2,3,C0003,DNI,38119160,Milagros,Villanueva Zevallos,milagros3@correo.com,911285064,Surco,Compra ocasional,2026-03-08,Ninguna
3,4,C0004,DNI,14282620,Karla,Ríos Ninaquispe,karla4@correo.com,999382396,Villa El Salvador,Convenio corporativo,2026-05-10,Ninguna
4,5,C0005,DNI,12191834,Hugo,Romero Castillo,hugo5@correo.com,982000067,Surco,Cliente frecuente,2026-04-20,Asma


**Pregunta 1:** ¿Cuántas tablas tiene la base y cuántos registros tiene `clientes`? ¿Por qué se dice que es una base *relacional*?

**Respuesta:** La base oficial tiene 8 tablas: `sedes`, `empleados`, `clientes`, `productos_servicios`, `operaciones`, `detalle_operacion`, `pagos` e `incidencias`. Se le llama relacional porque las tablas están conectadas entre sí mediante claves -FK  y PK - (por ejemplo `id_cliente` e `id_empleado` en `operaciones`), de modo que una tabla depende y hace referencia a registros de otra en lugar de tener toda la información repetida en una sola tabla

### Paso 4: diagnóstico de CALIDAD de la tabla clientes
Antes de limpiar, medimos el daño. Revisamos valores nulos, documentos (DNI) duplicados y correos mal escritos.

In [4]:
# Paso 4: diagnostico de calidad de clientes
print("Nulos por columna:")
display(df_clientes.isna().sum())

dup_doc = int(df_clientes["num_documento"].duplicated().sum())
correo = df_clientes["correo"].dropna()
correos_malos = int((~correo.str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)).sum())

print("\nDocumentos (num_documento) duplicados:", dup_doc)
print("Correos con formato invalido:", correos_malos)
print("Filas completamente duplicadas:", int(df_clientes.duplicated().sum()))

Nulos por columna:


,0
id_cliente,0
codigo_cliente,0
tipo_documento,0
num_documento,0
nombres,0
apellidos,0
correo,14
telefono,18
distrito,10
segmento,0



Documentos (num_documento) duplicados: 6
Correos con formato invalido: 11
Filas completamente duplicadas: 0


**Pregunta 2:** ¿Qué problemas de calidad observas en `clientes` antes de transformar?

**Respuesta:** Los problemas identificados fueron

*    `clientes` presenta 6 documentos (`num_documento`) duplicados
*   14 correos nulos
*   18 teléfonos nulos
*   11 correos con formato inválido
*   10 distritos nulos


### Paso 5: diagnóstico de INTEGRIDAD y montos
En una base relacional también fallan las **relaciones**: ventas que apuntan a un cliente o empleado que no existe (registros *huérfanos*) y montos imposibles (negativos). Los detectamos con `LEFT JOIN`.

In [5]:
# Paso 5: diagnostico de integridad referencial y montos
op_sin_cliente = pd.read_sql_query("""
    SELECT COUNT(*) AS n FROM operaciones o
    LEFT JOIN clientes c ON o.id_cliente = c.id_cliente
    WHERE c.id_cliente IS NULL""", conn_origen)["n"][0]
op_sin_empleado = pd.read_sql_query("""
    SELECT COUNT(*) AS n FROM operaciones o
    LEFT JOIN empleados e ON o.id_empleado = e.id_empleado
    WHERE e.id_empleado IS NULL""", conn_origen)["n"][0]
montos_negativos = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM operaciones WHERE monto_total < 0", conn_origen)["n"][0]

print("Ventas huerfanas (sin cliente valido):", op_sin_cliente)
print("Ventas huerfanas (sin empleado valido):", op_sin_empleado)
print("Ventas con monto_total negativo:", montos_negativos)

Ventas huerfanas (sin cliente valido): 15
Ventas huerfanas (sin empleado valido): 22
Ventas con monto_total negativo: 8


**Pregunta 3:** ¿Por qué un registro *huérfano* es un problema en una base relacional? Da un ejemplo con Farmacia MediSur.

**Respuesta:** Un registro huérfano es un problema porque rompe la integridad referencial: existe una fila que apunta a una clave (`id_cliente` o `id_empleado`) que ya no existe en la tabla relacionada, por lo que esa operación queda "sin dueño" y no se puede analizar ni auditar correctamente.

Asímismo, se detectaron 15 ventas sin un `id_cliente` válido y 22 ventas sin un `id_empleado` válido; por ejemplo, una venta huérfana sin cliente impide saber a quién facturarle o quién fue el vendedor.

### Paso 6: TRANSFORM — limpiar la tabla clientes con una función
Ponemos **todas** las limpiezas dentro de una función reutilizable. Así el mismo pipeline servirá para cualquier fuente nueva (Paso 11).

In [6]:
# Paso 6: TRANSFORM - limpieza de clientes en una funcion reutilizable
def transformar_clientes(df):
    df = df.copy()

    # 1. Eliminar documentos (DNI) duplicados, conservando el primero
    df = df.drop_duplicates(subset=["num_documento"], keep="first")

    # 2. Estandarizar texto (espacios y may/min)
    for col in ["nombres", "apellidos", "distrito", "segmento", "tipo_documento"]:
        df[col] = df[col].astype("string").str.strip().str.title()

    # 3. Corregir correos invalidos: los que no tienen formato valido se anulan
    formato_ok = df["correo"].str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)
    df["correo_fue_corregido"] = df["correo"].notna() & ~formato_ok
    df.loc[~formato_ok, "correo"] = pd.NA

    # 4. Completar valores faltantes con marcas claras
    df["correo"] = df["correo"].fillna("SIN CORREO REGISTRADO")
    df["telefono"] = df["telefono"].fillna("000000000")
    df["distrito"] = df["distrito"].fillna("No Especificado")

    # 5. Validar la fecha de registro y crear indicador de calidad
    df["fecha_registro_dt"] = pd.to_datetime(df["fecha_registro"], errors="coerce",
                                             format="mixed", dayfirst=True)
    df["fecha_valida"] = df["fecha_registro_dt"].notna()

    # 6. condicion_cronica es un DATO SENSIBLE (salud): se conserva, se gobierna en la S14
    return df

df_clientes_limpio = transformar_clientes(df_clientes)
print("Clientes antes :", len(df_clientes))
print("Clientes despues:", len(df_clientes_limpio), "(se quitaron DNI duplicados)")
display(df_clientes_limpio.head())

Clientes antes : 208
Clientes despues: 202 (se quitaron DNI duplicados)


,id_cliente,codigo_cliente,tipo_documento,num_documento,nombres,apellidos,correo,telefono,distrito,segmento,fecha_registro,condicion_cronica,correo_fue_corregido,fecha_registro_dt,fecha_valida
0,1,C0001,Ruc,79941052,Iván,Aguirre Bautista,iván1@correo.com,923380330,Rímac,Cliente Frecuente,2026-01-24,Hipertensión,False,2026-01-24,True
1,2,C0002,Dni,377593,Melissa,Torres Lévano,melissa2@correo.com,966971049,Villa El Salvador,Paciente Crónico,2026-05-18,Asma,False,2026-05-18,True
2,3,C0003,Dni,38119160,Milagros,Villanueva Zevallos,milagros3@correo.com,911285064,Surco,Compra Ocasional,2026-03-08,Ninguna,False,2026-03-08,True
3,4,C0004,Dni,14282620,Karla,Ríos Ninaquispe,karla4@correo.com,999382396,Villa El Salvador,Convenio Corporativo,2026-05-10,Ninguna,False,2026-05-10,True
4,5,C0005,Dni,12191834,Hugo,Romero Castillo,hugo5@correo.com,982000067,Surco,Cliente Frecuente,2026-04-20,Asma,False,2026-04-20,True


**Pregunta 4:** ¿Qué transformaciones aplicó la función y por qué son necesarias? ¿Por qué `condicion_cronica` no se elimina?

**Respuesta:** La función `transformar_clientes` aplica:
* (1) eliminación de los 6 DNI duplicados con `drop_duplicates`, necesaria para que cada cliente sea único.
* (2) estandarización de texto (`strip` y `title`) en nombres, apellidos, distrito, segmento y tipo de documento, necesaria para que los valores sean comparables y no encontrar "lima"/"LIMA"/"Lima.
* (3) anulación y marcado de los 11 correos con formato inválido, para no cargar datos de contacto incorrectos.
* (4) relleno de nulos en correo, teléfono y distrito con marcas explícitas ("SIN CORREO REGISTRADO", "000000000", "No Especificado"), esto con el fin de no perder el registro completo por un solo dato faltante.
* (5) validación del formato de `fecha_registro` creando un indicador `fecha_valida`.

La columna `condicion_cronica` no se elimina ni se modifica porque es un dato sensible de salud: se conserva para no perder información clínica relevante.

### Paso 7: TRANSFORM — corregir la tabla operaciones (ventas)
Corregimos los montos negativos (a cero, marcando el ajuste) y marcamos las ventas huérfanas para no perder trazabilidad.

In [7]:
# Paso 7: TRANSFORM - corregir operaciones
df_op = pd.read_sql_query("SELECT * FROM operaciones", conn_origen)
ids_cliente = set(pd.read_sql_query("SELECT id_cliente FROM clientes", conn_origen)["id_cliente"])
ids_empleado = set(pd.read_sql_query("SELECT id_empleado FROM empleados", conn_origen)["id_empleado"])

def transformar_operaciones(df):
    df = df.copy()
    df["monto_original"] = df["monto_total"]
    df["monto_ajustado"] = np.where(df["monto_total"] < 0, 0, df["monto_total"])
    df["monto_fue_ajustado"] = df["monto_total"] < 0
    df["cliente_valido"] = df["id_cliente"].isin(ids_cliente)
    df["empleado_valido"] = df["id_empleado"].isin(ids_empleado)
    return df

df_op_limpio = transformar_operaciones(df_op)
print("Ventas procesadas:", len(df_op_limpio))
print("Montos ajustados:", int(df_op_limpio["monto_fue_ajustado"].sum()))
print("Ventas marcadas sin cliente valido:", int((~df_op_limpio["cliente_valido"]).sum()))
print("Ventas marcadas sin empleado valido:", int((~df_op_limpio["empleado_valido"]).sum()))
display(df_op_limpio.head())

Ventas procesadas: 1010
Montos ajustados: 8
Ventas marcadas sin cliente valido: 15
Ventas marcadas sin empleado valido: 22


,id_operacion,codigo_operacion,id_cliente,id_sede,id_empleado,fecha_operacion,canal,estado,monto_total,monto_original,monto_ajustado,monto_fue_ajustado,cliente_valido,empleado_valido
0,1,O00001,69,4,11.0,2026-06-04,Mostrador,Completado,148.35,148.35,148.35,False,True,True
1,2,O00002,139,3,40.0,2026-05-01,Web,Observado,326.05,326.05,326.05,False,True,True
2,3,O00003,79,3,26.0,2026-02-26,App MediSur,Completado,792.22,792.22,792.22,False,True,True
3,4,O00004,89,3,3.0,2026-03-27,Delivery telefónico,Observado,400.38,400.38,400.38,False,True,True
4,5,O00005,79,6,6.0,2026-06-27,Mostrador,Cancelado,-1767.60,-1767.60,0.00,True,True,True


### Paso 8: resumen de transformaciones

In [8]:
# Paso 8: resumen de las transformaciones aplicadas
correo = df_clientes["correo"].dropna()
correos_malos = int((~correo.str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)).sum())

resumen_etl = pd.DataFrame([
    ["Clientes: DNI duplicados eliminados", int(df_clientes["num_documento"].duplicated().sum())],
    ["Clientes: correos nulos completados", int(df_clientes["correo"].isna().sum())],
    ["Clientes: telefonos nulos completados", int(df_clientes["telefono"].isna().sum())],
    ["Clientes: distritos nulos completados", int(df_clientes["distrito"].isna().sum())],
    ["Clientes: correos invalidos corregidos", correos_malos],
    ["Ventas: montos negativos ajustados", int(df_op_limpio["monto_fue_ajustado"].sum())],
    ["Ventas: huerfanas sin cliente", int((~df_op_limpio["cliente_valido"]).sum())],
    ["Ventas: huerfanas sin empleado", int((~df_op_limpio["empleado_valido"]).sum())],
], columns=["control", "cantidad"])

display(resumen_etl)

,control,cantidad
0,Clientes: DNI duplicados eliminados,6
1,Clientes: correos nulos completados,14
2,Clientes: telefonos nulos completados,18
3,Clientes: distritos nulos completados,10
4,Clientes: correos invalidos corregidos,11
5,Ventas: montos negativos ajustados,8
6,Ventas: huerfanas sin cliente,15
7,Ventas: huerfanas sin empleado,22


### Paso 9: LOAD — cargar los datos limpios a una nueva base
Guardamos las tablas limpias en `farmacia_migrada.db`. La base original **no se toca**: la migración siempre escribe en un destino nuevo.

In [9]:
# Paso 9: LOAD - cargar a la base destino
conn_destino = sqlite3.connect(DB_DESTINO)

df_clientes_limpio.to_sql("clientes_limpio", conn_destino, if_exists="replace", index=False)
df_op_limpio.to_sql("ventas_limpia", conn_destino, if_exists="replace", index=False)
resumen_etl.to_sql("resumen_etl", conn_destino, if_exists="replace", index=False)
conn_destino.commit()

print("Datos cargados en la base destino.")
display(pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn_destino))

Datos cargados en la base destino.


,name
0,clientes_limpio
1,resumen_etl
2,ventas_limpia


**Pregunta 5:** ¿Por qué la migración escribe en una base **nueva** en lugar de modificar la original?

**Respuesta:** Porque la base original debe conservarse como respaldo y evidencia de la fuente legacy. Si el pipeline modificara directamente `farmacia_legacy.db` y hubiera un error, se perdería la posibilidad de comparar origen vs. destino y de volver a ejecutar la migración desde cero.

### Paso 10: validación post-migración
Comparamos origen y destino: los clientes del destino deben ser los del origen menos los DNI duplicados.

In [10]:
# Paso 10: validacion post-migracion
clientes_origen = len(df_clientes)
clientes_destino = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM clientes_limpio", conn_destino)["n"][0]
dup_eliminados = int(df_clientes["num_documento"].duplicated().sum())

print("Clientes en origen :", clientes_origen)
print("Clientes en destino:", clientes_destino)
print("DNI duplicados eliminados:", dup_eliminados)
print("Validacion de conteo:", clientes_origen - dup_eliminados == clientes_destino)

validacion = pd.DataFrame([
    ["clientes_origen", clientes_origen],
    ["clientes_destino", clientes_destino],
    ["dni_duplicados_eliminados", dup_eliminados],
    ["correos_corregidos_en_destino",
     int(pd.read_sql_query("SELECT correo_fue_corregido FROM clientes_limpio", conn_destino)["correo_fue_corregido"].sum())],
    ["ventas_monto_ajustado",
     int(pd.read_sql_query("SELECT monto_fue_ajustado FROM ventas_limpia", conn_destino)["monto_fue_ajustado"].sum())],
], columns=["indicador", "valor"])

display(validacion)

Clientes en origen : 208
Clientes en destino: 202
DNI duplicados eliminados: 6
Validacion de conteo: True


,indicador,valor
0,clientes_origen,208
1,clientes_destino,202
2,dni_duplicados_eliminados,6
3,correos_corregidos_en_destino,11
4,ventas_monto_ajustado,8


**Pregunta 6:** ¿La migración fue correcta? Justifica con los conteos de origen y destino.

**Respuesta:** Sí, la migración fue correcta. La tabla `clientes` en origen tenía 208 registros y en destino (`clientes_limpio`) quedó con 202, esta diferencia corresponde a los DNI duplicados eliminados (208 - 6 = 202).

Además, se validó que los 11 correos inválidos quedaran marcados como corregidos y que las 8 ventas con monto negativo quedaran ajustadas a 0 en `ventas_limpia`.

### Paso 11: segunda fuente — una sucursal aliada llega en CSV
La cadena sumó una **sucursal aliada** que exporta a sus clientes con **otros nombres de columna** (`DOC`, `NOMBRE`, `APELLIDO`, `EMAIL`, `CEL`, `ZONA`). Antes de unificarla al destino, hay que renombrar sus columnas al esquema estándar.

In [11]:
# Paso 11: generar el CSV de la sucursal aliada (fuente nueva con columnas distintas)
np.random.seed(11)
nombres = ["Ana Torres", "Luis Ramirez", "Carmen Vega", "Marco Huaman",
           "Rosa Castillo", "Diego Flores", "Julia Paredes", "Pedro Quispe"]
filas_ext = []
for i in range(1, 26):
    nom = np.random.choice(nombres).split()
    correo = f"{nom[0].lower()}@correo.com" if np.random.rand() > 0.1 else "correo_malo"
    filas_ext.append([9000 + i, f"C9{i:04d}", "DNI", f"7{i:07d}",
                      nom[0], nom[1], correo,
                      f"9{np.random.randint(10000000, 99999999)}",
                      np.random.choice(["Surco", "  Ate ", "lince", None]),
                      "Nuevo", "2026-05-15", "Ninguna"])

df_ext = pd.DataFrame(filas_ext, columns=[
    "id_cliente", "codigo_cliente", "tipo_documento", "DOC", "NOMBRE", "APELLIDO",
    "EMAIL", "CEL", "ZONA", "segmento", "fecha_registro", "condicion_cronica"])
df_ext.to_csv("clientes_sucursal_aliada.csv", index=False)

df_nueva = pd.read_csv("clientes_sucursal_aliada.csv")
mapa = {"DOC": "num_documento", "NOMBRE": "nombres", "APELLIDO": "apellidos",
        "EMAIL": "correo", "CEL": "telefono", "ZONA": "distrito"}
df_nueva = df_nueva.rename(columns=mapa)
print("Columnas unificadas:", list(df_nueva.columns))
display(df_nueva.head())

Columnas unificadas: ['id_cliente', 'codigo_cliente', 'tipo_documento', 'num_documento', 'nombres', 'apellidos', 'correo', 'telefono', 'distrito', 'segmento', 'fecha_registro', 'condicion_cronica']


,id_cliente,codigo_cliente,tipo_documento,num_documento,nombres,apellidos,correo,telefono,distrito,segmento,fecha_registro,condicion_cronica
0,9001,C90001,DNI,70000001,Luis,Ramirez,correo_malo,946379739,Ate,Nuevo,2026-05-15,Ninguna
1,9002,C90002,DNI,70000002,Pedro,Quispe,pedro@correo.com,969930273,NaN,Nuevo,2026-05-15,Ninguna
2,9003,C90003,DNI,70000003,Carmen,Vega,carmen@correo.com,989979184,Ate,Nuevo,2026-05-15,Ninguna
3,9004,C90004,DNI,70000004,Ana,Torres,correo_malo,973171197,Surco,Nuevo,2026-05-15,Ninguna
4,9005,C90005,DNI,70000005,Carmen,Vega,carmen@correo.com,991192778,Ate,Nuevo,2026-05-15,Ninguna


**Pregunta 7:** ¿Qué pasaría si cargáramos la fuente nueva **sin** renombrar sus columnas? ¿Por qué la migración exige un esquema común?

**Respuesta:** Si se carga el CSV de la sucursal aliada sin renombrar columnas como `DOC`, `NOMBRE`, `APELLIDO`, `EMAIL`, `CEL` y `ZONA`, el pipeline fallaría o generaría columnas nuevas en la tabla destino, ya que la función `transformar_clientes` espera nombres como `num_documento`, `nombres`, `apellidos`, `correo`, `telefono` y `distrito`. Esto rompería el proceso de limpieza.

Por eso la migración exige un esquema común: solo así los datos de distintas fuentes son comparables, se pueden unir de forma consistente, el mismo pipeline es reutilizable  y funciona para cualquier sucursal nueva.

### Paso 12: función reutilizable — un stored procedure simulado
En SQL Server o PostgreSQL esto sería un **stored procedure** (`CREATE PROCEDURE`). En SQLite lo simulamos con una función de Python que ejecuta el pipeline completo cada vez que llega una fuente nueva.

In [12]:
# Paso 12: funcion reutilizable - stored procedure simulado
def cargar_clientes(df_fuente, conn, tabla="clientes_limpio"):
    """Limpia una fuente con el pipeline del Paso 6 y la agrega a la tabla destino."""
    df = transformar_clientes(df_fuente)
    df.to_sql(tabla, conn, if_exists="append", index=False)
    return len(df)

n_aliada = cargar_clientes(df_nueva, conn_destino)
print("Clientes de la sucursal aliada cargados:", n_aliada)

Clientes de la sucursal aliada cargados: 25


**Pregunta 8:** ¿Qué ventaja tiene envolver el pipeline en una función reutilizable? ¿Qué relación tiene con un stored procedure?

**Respuesta:** Envolver la limpieza en la función `cargar_clientes` (que también usa `transformar_clientes`) permite aplicar exactamente la misma lógica de limpieza y validación cada vez que llega una fuente nueva, sin reescribir código y sin riesgo de que una sucursal se limpie distinto a otra.

Esto es equivalente en esencia a un *stored procedure*, porque en ambos casos se encapsula un proceso repetible (transformar y cargar datos) que se invoca con un parámetro (la fuente nueva).

### Paso 13: checklist de validación de la Semana 10

In [13]:
# Paso 13: checklist de validacion
destino = pd.read_sql_query("SELECT * FROM clientes_limpio", conn_destino)
total_destino = len(destino)
esperado = (clientes_origen - dup_eliminados) + n_aliada

checklist = pd.DataFrame([
    ["Registros esperados = registros en destino", bool(esperado == total_destino)],
    ["Cero nulos en columnas clave (nombres, correo)", bool(destino[["nombres", "correo"]].isna().sum().sum() == 0)],
    ["Sin DNI duplicados dentro de la base migrada original", bool(df_clientes_limpio["num_documento"].duplicated().sum() == 0)],
    ["Correo nunca queda vacio (usa marca)", bool((destino["correo"].str.strip() != "").all())],
    ["Distrito nunca queda nulo", bool(destino["distrito"].isna().sum() == 0)],
], columns=["verificacion", "cumple"])

display(checklist)

,verificacion,cumple
0,Registros esperados = registros en destino,True
1,"Cero nulos en columnas clave (nombres, correo)",True
2,Sin DNI duplicados dentro de la base migrada o...,True
3,Correo nunca queda vacio (usa marca),True
4,Distrito nunca queda nulo,True


**Pregunta 9 (reto):** El pipeline dejó pasar un problema: al **unir** la base original con la sucursal aliada podría haber **DNI repetidos entre ambas fuentes**. Detéctalo con `destino["num_documento"].value_counts()` y explica cómo lo corregirías.

**Respuesta:** Al ejecutar `destino["num_documento"].value_counts()` sobre los 227 registros finales no se encontraron DNI repetidos entre ambas fuentes en esta ejecución.

Sin embargo,la función `cargar_clientes` usa `to_sql(..., if_exists="append")`, que simplemente agrega las filas nuevas a clientes_limpio sin comparar contra los DNI que ya están cargados ahí.

Antes de cargar cualquier fuente nueva, el pipeline debería primero consultar qué documentos ya existen en la tabla destino clientes_limpio y comparar la fuente nueva contra ese conjunto completo

### Paso 14: reporte para el negocio y exportación

In [14]:
# Paso 14: reporte para el negocio, exportación y cierre

import os
import zipfile
from google.colab import files

# 1. Reporte para el negocio
reporte = pd.read_sql_query("""
    SELECT
        distrito,
        segmento,
        COUNT(*) AS total_clientes
    FROM clientes_limpio
    GROUP BY distrito, segmento
    ORDER BY total_clientes DESC
""", conn_destino)

display(reporte.head(10))

# 2. Volver a leer todos los clientes de la base destino
# Así también se incluyen los clientes de la sucursal aliada
destino_final = pd.read_sql_query(
    "SELECT * FROM clientes_limpio",
    conn_destino
)

# 3. Rutas de los archivos
archivos = {
    "clientes": "/content/clientes_limpios_farmacia.csv",
    "resumen": "/content/resumen_etl_farmacia.csv",
    "validacion": "/content/validacion_etl_farmacia.csv",
    "checklist": "/content/checklist_etl_farmacia.csv",
    "reporte": "/content/reporte_clientes_farmacia.csv"
}

# 4. Exportar los DataFrames a CSV
destino_final.to_csv(
    archivos["clientes"],
    index=False,
    encoding="utf-8-sig"
)

resumen_etl.to_csv(
    archivos["resumen"],
    index=False,
    encoding="utf-8-sig"
)

validacion.to_csv(
    archivos["validacion"],
    index=False,
    encoding="utf-8-sig"
)

checklist.to_csv(
    archivos["checklist"],
    index=False,
    encoding="utf-8-sig"
)

reporte.to_csv(
    archivos["reporte"],
    index=False,
    encoding="utf-8-sig"
)

# 5. Confirmar los cambios antes de cerrar
conn_destino.commit()

# 6. Cerrar las conexiones
conn_origen.close()
conn_destino.close()

# 7. Agregar la base de datos a la lista
ruta_base = DB_DESTINO

# 8. Crear un archivo ZIP
ruta_zip = "/content/entrega_etl_farmacia.zip"

with zipfile.ZipFile(
    ruta_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    # Agregar los archivos CSV
    for ruta in archivos.values():
        if os.path.exists(ruta):
            zipf.write(
                ruta,
                arcname=os.path.basename(ruta)
            )

    # Agregar la base SQLite
    if os.path.exists(ruta_base):
        zipf.write(
            ruta_base,
            arcname="farmacia_migrada.db"
        )

# 9. Mostrar los archivos generados
print("Archivos generados:")

for ruta in archivos.values():
    if os.path.exists(ruta):
        print("  ✓", os.path.basename(ruta))
    else:
        print("  ✗ No encontrado:", ruta)

if os.path.exists(ruta_base):
    print("  ✓ farmacia_migrada.db")
else:
    print("  ✗ No se encontró la base:", ruta_base)

print("\nLaboratorio finalizado correctamente.")
print("Descargando:", os.path.basename(ruta_zip))

# 10. Descargar todos los archivos juntos
files.download(ruta_zip)

,distrito,segmento,total_clientes
0,No Especificado,Nuevo,9
1,Ate,Cliente Frecuente,7
2,Lince,Convenio Corporativo,6
3,Los Olivos,Cliente Frecuente,6
4,Miraflores,Compra Ocasional,6
5,Miraflores,Paciente Crónico,6
6,Rímac,Cliente Frecuente,6
7,Rímac,Paciente Crónico,6
8,Surco,Compra Ocasional,6
9,Surco,Nuevo,6


Archivos generados:
  ✓ clientes_limpios_farmacia.csv
  ✓ resumen_etl_farmacia.csv
  ✓ validacion_etl_farmacia.csv
  ✓ checklist_etl_farmacia.csv
  ✓ reporte_clientes_farmacia.csv
  ✓ farmacia_migrada.db

Laboratorio finalizado correctamente.
Descargando: entrega_etl_farmacia.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Actividad 3: caso de estudio — cierre de la migración MediSur

**Pregunta A:** ¿Qué problemas de calidad e integridad traía la base oficial de Farmacia MediSur y cuáles corrigió tu pipeline?

**Respuesta:** La base traía problemas de calidad (6 DNI duplicados, 14 correos nulos, 11 correos con formato inválido, 18 teléfonos nulos y 10 distritos nulos) y problemas de integridad (8 ventas con monto negativo, 15 ventas huérfanas sin cliente válido y 22 ventas huérfanas sin empleado válido). El pipeline corrigió todos estos casos.

**Pregunta B:** ¿Qué controles de validación aplicaste y qué resultados obtuviste? Compara los conteos de origen y destino.

**Respuesta:** Se aplicaron cuatro controles principales:
* (1) conteo de registros, origen 208 vs. destino 202
* (2) checklist de nulos en columnas clave (nombres, correo), que dio 0 nulos en destino.
* (3) verificación de unicidad de `num_documento` en la base migrada original, sin duplicados.
* (4) validación de que el total esperado tras sumar la sucursal aliada (202 + 25 = 227).

**Pregunta C:** Muestra el antes y después con `df.shape`. ¿Hubo pérdida intencional de datos (duplicados)? Justifica.

**Respuesta:** `df_clientes.shape` antes de transformar es (208, 12) y `df_clientes_limpio.shape` después es (202, 14) —se agregaron columnas de control como `correo_fue_corregido` y `fecha_valida`, y se perdieron 6 filas—.

Sí hubo pérdida intencional de datos: los 6 registros eliminados corresponden a DNI duplicados, y se conservó únicamente el primer registro de cada uno con `keep="first"`.

## Actividad final

Redacta tres conclusiones breves:

1. **¿Qué aprendiste sobre ETL y migración de datos?** Aprendí que un pipeline ETL no es solo mover datos de un lugar a otro: la fase Transform es la más crítica porque ahí se decide cómo tratar duplicados, nulos y errores de formato, y que ninguna migración debe considerarse exitosa solo porque el código corrió sin errores; siempre hay que validarla comparando origen y destino.

2. **¿Qué problema de calidad o de integridad te pareció el más importante?** Los registros huérfanos en `operaciones` (15 sin cliente, 22 sin empleado) me parecieron los más importantes, porque pueden causar impacto directo en la facturación, el control interno de la farmacia y una posible auditoria.

3. **¿Cómo te servirá esta base limpia para las semanas siguientes?** El proceso de limpieza etl me servirá para tratar la base de datos que se trabajará para el proyecto final, así se podrá asegurar que los datos se encuentren listos para poderlos trabajar.